# 5. Explainability

**Нужны:** все файлы из тетрадок 1–3

**Создаёт:** `recommendations_explained.csv`

In [1]:
import csv, math, os, random
from collections import defaultdict, Counter

random.seed(42)
BASE_DIR = os.path.dirname(os.path.abspath("__file__"))

def load_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def parse_vector(s):
    if not s or not s.strip(): return None
    return [float(x) for x in s.split(",")]

def parse_set(s):
    if not s or not s.strip(): return None
    return set(s.split("|"))

def parse_bool(s):
    return str(s).strip().lower() in ("true","1","yes")

DOMAINS = [r["domain_name"].strip()
           for r in load_csv(os.path.join(BASE_DIR,"ontology_domains.csv"))]

SIM_MATRIX = {}
with open(os.path.join(BASE_DIR,"ontology_domain_similarity.csv"),
          newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        d1 = row["domain"].strip()
        for d2 in DOMAINS:
            SIM_MATRIX[(d1,d2)] = float(row.get(d2,0))

ratings_by_id = {r["mentor_id"]: float(r["rating"])
                 for r in load_csv(os.path.join(BASE_DIR,"mentor_ratings.csv"))}

print(f"Областей: {len(DOMAINS)}  |  Рейтингов: {len(ratings_by_id)}")


Областей: 10  |  Рейтингов: 2000


In [2]:
def parse_mentee(r):
    return {"id":r["id"],"name":r["name"],
            "level_score":int(r["level_score"]),"level_raw":r.get("level_raw",""),
            "domain":r.get("domain",""),"profession":r.get("profession",""),
            "skills_missing":parse_bool(r["skills_missing"]),
            "language_set":parse_set(r["language_set"]),"format_set":parse_set(r["format_set"]),
            "domain_vector":parse_vector(r["domain_vector"]),
            "skills_list":[s.strip() for s in r.get("skills_normalized","").split(";") if s.strip()]}

def parse_mentor(r):
    return {"id":r["id"],"name":r["name"],"profession":r["profession"],"domain":r.get("domain",""),
            "level_score":int(r["level_score"]),"level_raw":r.get("level_raw",""),
            "language_set":parse_set(r["language_set"]),"format_set":parse_set(r["format_set"]),
            "domain_vector":parse_vector(r["domain_vector"]),
            "skills_list":[s.strip() for s in r.get("skills_normalized","").split(";") if s.strip()],
            "experience_years":int(r.get("experience_years",0) or 0),
            "experience_norm":float(r["experience_norm"]),
            "available":parse_bool(r["available"]),
            "boosted":parse_bool(r["boosted"]),"boost_k":float(r["boost_k"]),
            "rating":ratings_by_id.get(r["id"],3.8)}

mentees_by_id = {m["id"]:m for m in [parse_mentee(r)
    for r in load_csv(os.path.join(BASE_DIR,"mentees_processed.csv"))]}
mentors_by_id = {m["id"]:m for m in [parse_mentor(r)
    for r in load_csv(os.path.join(BASE_DIR,"mentors_processed.csv"))]}
weights_by_id = {w["mentee_id"]:w
    for w in load_csv(os.path.join(BASE_DIR,"mentee_weights.csv"))}
bounds = {r["factor"]:r for r in load_csv(os.path.join(BASE_DIR,"normalization_bounds.csv"))}
SKILL_MIN,SKILL_MAX   = float(bounds["skill"]["min"]),  float(bounds["skill"]["max"])
DOMAIN_MIN,DOMAIN_MAX = float(bounds["domain"]["min"]), float(bounds["domain"]["max"])
EXP_MIN,EXP_MAX       = float(bounds["exp"]["min"]),    float(bounds["exp"]["max"])
RATING_MIN,RATING_MAX = float(bounds["rating"]["min"]), float(bounds["rating"]["max"])
print(f"Загружено: {len(mentees_by_id)} менти, {len(mentors_by_id)} менторов")


Загружено: 5000 менти, 2000 менторов


In [3]:
# ── 4 фактора: skill, domain, exp, rating (goal убран — вклад <0.5%) ─────────

def jaccard(s1, s2):
    a, b = set(s1), set(s2)
    if not a or not b: return None
    return len(a & b) / len(a | b)

def domain_sim_raw(mentee, mentor):
    v1, v2 = mentee["domain_vector"], mentor["domain_vector"]
    if not v1 or not v2: return 0.0
    return sum(
        v1[i] * SIM_MATRIX.get((d1,d2), 0) * v2[j]
        for i,d1 in enumerate(DOMAINS)
        for j,d2 in enumerate(DOMAINS)
    )

def sets_compat(s1, s2):
    if s1 is None or s2 is None: return True
    return len(s1 & s2) > 0

def hard_filter(mentee, pool):
    return [m for m in pool
            if m["available"]
            and m["level_score"] > mentee["level_score"]
            and sets_compat(mentee["language_set"], m["language_set"])
            and sets_compat(mentee["format_set"],   m["format_set"])]

def minmax(v, vmin, vmax):
    if vmax == vmin: return 0.5
    return round(max(0.0, min(1.0, (v - vmin) / (vmax - vmin))), 4)

def compute_score(mentee, mentor, weights):
    # skill: Jaccard. Пустой профиль → 0 (наказание, не исключение)
    sk_raw = jaccard(mentee["skills_list"], mentor["skills_list"])
    do_raw = domain_sim_raw(mentee, mentor)

    skill_val  = minmax(sk_raw, SKILL_MIN, SKILL_MAX) if sk_raw is not None else 0.0
    domain_val = minmax(do_raw, DOMAIN_MIN, DOMAIN_MAX)
    exp_val    = minmax(mentor["experience_norm"], EXP_MIN, EXP_MAX)
    rating_val = minmax((mentor["rating"] - 1) / 4, RATING_MIN, RATING_MAX)

    w_sk = float(weights["w_skills"])
    w_do = float(weights["w_domain"])
    w_ex = float(weights["w_exp"])
    w_ra = float(weights["w_rating"])

    score = w_sk*skill_val + w_do*domain_val + w_ex*exp_val + w_ra*rating_val

    breakdown = {
        "skill":  {"sim":skill_val,  "weight":w_sk,
                   "contribution":round(w_sk*skill_val,4),
                   "penalized": sk_raw is None},
        "domain": {"sim":domain_val, "weight":w_do,
                   "contribution":round(w_do*domain_val,4), "penalized":False},
        "exp":    {"sim":exp_val,    "weight":w_ex,
                   "contribution":round(w_ex*exp_val,4),    "penalized":False},
        "rating": {"sim":rating_val, "weight":w_ra,
                   "contribution":round(w_ra*rating_val,4), "penalized":False},
    }
    return round(score, 4), breakdown

BOOST_THRESHOLD = 0.30
TOP_K = 5

def apply_boost(score, mentor):
    if mentor["boosted"] and score >= BOOST_THRESHOLD:
        return round(score * (1 + mentor["boost_k"]), 4), True
    return score, False

print("Функции скоринга определены (4 фактора: skill, domain, exp, rating)")


Функции скоринга определены (4 фактора: skill, domain, exp, rating)


In [4]:
LEVEL_LBL = {1:"Junior",2:"Middle",3:"Senior",4:"Lead"}

def top_domain(vec):
    if vec is None: return "неизвестно"
    return DOMAINS[max(range(len(DOMAINS)),key=lambda i:vec[i])]

def domain_label(sim):
    if sim>=0.70: return "очень близкие области"
    if sim>=0.50: return "смежные области"
    if sim>=0.30: return "частично пересекаются"
    return "разные области"

def explain_main(breakdown, mentee, mentor):
    if any(v.get("penalized") for v in breakdown.values()):
        pct = breakdown["skill"]["weight"]*100
        return f"Профиль неполный — навыки не указаны. Score снижен на {pct:.0f}%."
    top = max(breakdown, key=lambda k:breakdown[k]["contribution"])
    ms  = sorted(set(mentee["skills_list"]) & set(mentor["skills_list"]))
    if top=="skill":
        return f"Совпадают навыки: {', '.join(ms[:3])}." if ms else "Схожий технический профиль."
    if top=="domain":
        d1,d2 = top_domain(mentee["domain_vector"]),top_domain(mentor["domain_vector"])
        return f"Оба из {d1}." if d1==d2 else f"{d1} и {d2} — {domain_label(domain_sim_raw(mentee,mentor))}."
    if top=="exp":
        return f"Большой опыт: {mentor['experience_years']} лет."
    return f"Высокий рейтинг: {mentor['rating']:.1f}/5.0."

recs = load_csv(os.path.join(BASE_DIR,"recommendations.csv"))
explained = []

for rec in recs:
    mentee  = mentees_by_id.get(rec["mentee_id"])
    mentor  = mentors_by_id.get(rec["mentor_id"])
    weights = weights_by_id.get(rec["mentee_id"])
    if not mentee or not mentor or not weights: continue

    organic, breakdown = compute_score(mentee, mentor, weights)
    is_boosted = mentor["boosted"] and organic >= BOOST_THRESHOLD
    ms = sorted(set(mentee["skills_list"]) & set(mentor["skills_list"]))
    un = sorted(set(mentor["skills_list"]) - set(mentee["skills_list"]))[:3]
    top_factor = max(breakdown, key=lambda k:breakdown[k]["contribution"])
    penalized  = any(v.get("penalized") for v in breakdown.values())

    do_raw = domain_sim_raw(mentee, mentor)
    d1,d2  = top_domain(mentee["domain_vector"]),top_domain(mentor["domain_vector"])

    expl_skill = (f"Навыки не указаны — штраф {breakdown['skill']['weight']*100:.0f}%."
                  if penalized else
                  (f"Совпадают {len(ms)}: {', '.join(ms[:4])}." if ms else "Прямых совпадений нет.") +
                  (f" Можно научиться: {', '.join(un)}." if un else ""))
    expl_domain = f"{d1} → {d2}: {domain_label(do_raw)} (схожесть {do_raw:.2f})."
    expl_exp    = (f"{mentor['experience_years']} лет — видел многое." if mentor['experience_years']>=10
                   else f"{mentor['experience_years']} лет — хорошая база.")
    expl_rating = (f"Рейтинг {mentor['rating']:.1f}/5.0 — один из лучших." if mentor['rating']>=4.5
                   else f"Рейтинг {mentor['rating']:.1f}/5.0.")

    explained.append({
        "mentee_id":rec["mentee_id"],"mentor_id":rec["mentor_id"],
        "rank":rec["rank"],"final_score":rec["final_score"],
        "is_boosted":is_boosted,"top_factor":top_factor,"profile_penalized":penalized,
        "main_reason":explain_main(breakdown,mentee,mentor),
        "matching_skills":"; ".join(ms),"learn_from_mentor":"; ".join(un),
        "explain_skill":expl_skill,"explain_domain":expl_domain,
        "explain_exp":expl_exp,"explain_rating":expl_rating,
    })

FIELDS = ["mentee_id","mentor_id","rank","final_score","is_boosted","top_factor",
          "profile_penalized","main_reason","matching_skills","learn_from_mentor",
          "explain_skill","explain_domain","explain_exp","explain_rating"]

with open(os.path.join(BASE_DIR,"recommendations_explained.csv"),"w",newline="",encoding="utf-8") as f:
    wcsv = csv.DictWriter(f, fieldnames=FIELDS)
    wcsv.writeheader()
    wcsv.writerows(explained)

n_pen = sum(1 for r in explained if r["profile_penalized"])
print(f"recommendations_explained.csv  ({len(explained)} строк)")
print(f"  Из них с штрафом: {n_pen} ({n_pen/len(explained)*100:.1f}%)")
print("Тетрадка 5 завершена!")


recommendations_explained.csv  (25000 строк)
  Из них с штрафом: 5334 (21.3%)
Тетрадка 5 завершена!
